## NOTES: ML deployment 

Machine learning model deployment: process of making a trained model available via a web service that can be accessed through other apps in real time. 

### Why deploy models?
- Real time predictions: enable other services to get outputs from models
- Scalability: handling of multiple requests
- Integration: allow different apps to use your model
- Production use: move from dev to prod --> business value 

## Workflow 

1. Model prep and saving in notebook 

Use Pickle:

In [ ]:
import pickle

# Save model and preprocessor
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('preprocessor.pkl', 'wb') as f:
    pickle.dump(dict_vectorizer, f)

## key parts to save:
- Trained Model
- Feasture transformation: preprocessers
- any preprocessing steps: scalers, encoders

# Ensure 
- Exact same dependencies must be installed in production
- Same library versions to avoid compatibility issues 
- Consistent Python environment

## 2. Create a webserver using Flask
Transform model into a network accessible service using flask
Benefits:
- simple to implement
- HTTP protocol using GET/POST
- Easy to test and debug
- Network accessibility

In [ ]:
from flask import Flask, request, jsonify
import pickle

app = Flask(__name__)

# Load model and preprocessor
with open('model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('preprocessor.pkl', 'rb') as f:
    dv = pickle.load(f)

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json()
    
    # Transform data
    X = dv.transform([data])
    
    # Make prediction
    prediction = model.predict_proba(X)[0, 1]
    
    return jsonify({'probability': float(prediction)})

if __name__ == '__main__':
    app.run(debug=True, host='0.0.0.0', port=9696)

## Dependency Management with pipenv:
Issue: Different projects need different library versions
Solve: Using virtual environment with pipenv

In [ ]:
# Install pipenv
pip install pipenv

# Create virtual environment and install dependencies
pipenv install flask scikit-learn pandas numpy

# Install development dependencies
pipenv install --dev pytest

# Activate environment
pipenv shell

# Run application
pipenv run python app.py

### Keep Pipenv files:
- Pipfile: human-readable dependency specification 
- Pipfile.lock: Exact versions and checksums for reproducibility 

### Why?
- Isolation: separate dependencies per project
- Reproducibility: locked versions ensure consistency 
- Security: checksums verify package integrity 
- Simplicty: easy to manage and share

Example pipefile:

In [ ]:
[[source]]
url = "https://pypi.org/simple"
verify_ssl = true
name = "pypi"

[packages]
flask = "*"
scikit-learn = "==1.6.1"
pandas = "*"

[dev-packages]
pytest = "*"

[requires]
python_version = "3.12"

## Containerisation with Docker 

### Why Docker?

Docker allows for completes system isolation:
- OS: specific linux distrbutions
- system libraries: compliers, system rools
- Complete application: everything needed to run 
- portabiltiy: runs anywhere supported

Benefits:
- Consistency: same environemnt everywhere 
- Portability: Deploy anywhere
- Scalability: easy to replicate

In [ ]:
FROM python:3.12-slim

WORKDIR /app

# Copy dependency files
COPY Pipfile Pipfile.lock ./

# Install pipenv and dependencies
RUN pip install pipenv && \
    pipenv install --system --deploy

# Copy application code
COPY app.py model.pkl preprocessor.pkl ./

# Expose port
EXPOSE 9696

# Run application
ENTRYPOINT ["python", "app.py"]

In [ ]:
# Build image
docker build -t ml-model-service .

# Run container
docker run -p 9696:9696 ml-model-service

# Test the service
curl -X POST http://localhost:9696/predict \
     -H "Content-Type: application/json" \
     -d '{"feature1": "value1", "feature2": 123}'

## CLOUD DEPLOYMENT 

## AWS Elastic Beanstalk 

A Platform-as-a-Service (PaaS) that simplifies deployment by automatically handling:
- Infrastructure set up: load balancers, auto-scaling groups
- Auto-scaling: automatically adjust capacity based on demand
- Config: environment variables and logging
- Security: basic security configs 

Benefits:
- automated infrastructure
- Autoscaling: can traffic spikes
- Load balancing: distribute requests across instances
- Monitoring: built-in health checks and metrics 

In [ ]:
# Install EB CLI
pip install awsebcli

# Initialize EB application
eb init

# Create environment and deploy
eb create ml-model-production

# Check status
eb status

# View logs
eb logs

# Terminate when done (important for cost!)
eb terminate

In [ ]:
#confidg example

In [ ]:
option_settings:
  aws:elasticbeanstalk:container:python:
    WSGIPath: app:app
  aws:elasticbeanstalk:application:environment:
    PYTHONPATH: "/var/app/current"

In [ ]:
option_settings:
  aws:elasticbeanstalk:container:python:
    WSGIPath: app:app
  aws:elasticbeanstalk:application:environment:
    PYTHONPATH: "/var/app/current"

## best practices for deploying ML models

1. Model Versioning

In [ ]:
# Include version in model filename
model_version = "v1.2.3"
pickle.dump(model, open(f'model_{model_version}.pkl', 'wb'))

2. Health checkpoint

In [ ]:
@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'healthy', 'version': '1.0.0'})

3. Error Handling

In [ ]:
@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json()
        prediction = model.predict_proba([data])[0, 1]
        return jsonify({'prediction': float(prediction)})
    except Exception as e:
        return jsonify({'error': str(e)}), 400

In [ ]:
## Logging:

import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

@app.route('/predict', methods=['POST'])
def predict():
    logger.info(f"Received prediction request: {request.get_json()}")
    # ... prediction logic ...
    logger.info(f"Returning prediction: {prediction}")

In [ ]:
## Configuration Management

import os

class Config:
    MODEL_PATH = os.environ.get('MODEL_PATH', 'model.pkl')
    DEBUG = os.environ.get('DEBUG', 'False').lower() == 'true'
    PORT = int(os.environ.get('PORT', 9696))

### 1. Model Monitoring and Observability
Once deployed, monitoring is crucial for production systems:

### Infrastructure Metrics:

📊 Latency: Response time monitoring
🖥️ Resource Utilization: CPU, memory, disk usage
📈 Scaling Events: Auto-scaling activity
🔗 Request Volume: Traffic patterns
### Model Performance Metrics:

📉 Data Drift: Changes in input data characteristics
🎯 Model Drift: Decrease in predictive accuracy over time
📊 Prediction Distribution: Output pattern changes
🔄 Feedback Loops: Actual vs predicted outcomes

In [ ]:
from prometheus_client import Counter, Histogram, generate_latest

REQUEST_COUNT = Counter('ml_requests_total', 'Total ML requests')
REQUEST_LATENCY = Histogram('ml_request_duration_seconds', 'Request latency')

@app.route('/predict', methods=['POST'])
def predict():
    REQUEST_COUNT.inc()
    with REQUEST_LATENCY.time():
        # prediction logic
        pass